# Merge Historical ERA5 Data (1971-2021) with New Data (2022-2024)

This notebook combines your existing ERA5-derived heat index data with the newly downloaded 2022-2024 data to create complete 1971-2024 datasets.

## What This Notebook Does:
1. Loads existing 1971-2021 ERA5 data
2. Loads new 2022-2024 ERA5 data
3. Checks for overlaps and data quality
4. Combines datasets
5. Performs quality checks
6. Saves complete 1971-2024 files

## Requirements:
- Existing 1971-2021 Excel files
- New 2022-2024 Excel files (from download notebook)
- Packages: `pandas`, `numpy`, `openpyxl`

## Estimated Time: 2-3 minutes

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime

print("✓ Packages imported")
print(f"pandas version: {pd.__version__}")

## 2. Configuration

**📝 Update these file paths to match your setup!**

In [ ]:
# Old data files (1971-2021)
# UPDATE THESE PATHS if your files are named differently or in a different location
OLD_DATA_FILES = {
    'KAVL': 'KAVLheatindex19712021.xlsx',
    'KGSO': 'KGSOheatindex19712021.xlsx',
    'KHSE': 'KHSEheatindex19712021.xlsx',
    'KILM': 'KILMheatindex19712021.xlsx',
    'KCLT': 'KLCTheatindex19712021.xlsx',  # Note: might be KLCT
    'KRDU': 'KRDUheatindex19712021.xlsx',
}

# New data files (2022-2024) - from download notebook
NEW_DATA_FILES = {
    'KAVL': 'KAVLheatindex20222024_era5.xlsx',
    'KGSO': 'KGSOheatindex20222024_era5.xlsx',
    'KHSE': 'KHSEheatindex20222024_era5.xlsx',
    'KILM': 'KILMheatindex20222024_era5.xlsx',
    'KCLT': 'KCLTheatindex20222024_era5.xlsx',
    'KRDU': 'KRDUheatindex20222024_era5.xlsx',
}

STATION_NAMES = {
    'KAVL': 'Asheville',
    'KGSO': 'Greensboro',
    'KHSE': 'Cape Hatteras',
    'KILM': 'Wilmington',
    'KCLT': 'Charlotte',
    'KRDU': 'Raleigh-Durham',
}

# Output folder
OUTPUT_FOLDER = 'merged_era5_data'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"✓ Configuration set")
print(f"  Stations: {len(STATION_NAMES)}")
print(f"  Output folder: {OUTPUT_FOLDER}")

## 3. Check File Availability

Verify that all required files exist before proceeding.

In [ ]:
print("Checking file availability...\n")

missing_files = []

for station_code in STATION_NAMES.keys():
    old_file = OLD_DATA_FILES[station_code]
    new_file = NEW_DATA_FILES[station_code]
    
    print(f"{STATION_NAMES[station_code]} ({station_code}):")
    
    if os.path.exists(old_file):
        print(f"  ✓ Old data: {old_file}")
    else:
        print(f"  ✗ Old data missing: {old_file}")
        missing_files.append(old_file)
    
    if os.path.exists(new_file):
        print(f"  ✓ New data: {new_file}")
    else:
        print(f"  ✗ New data missing: {new_file}")
        missing_files.append(new_file)

if missing_files:
    print(f"\n⚠️ Warning: {len(missing_files)} files are missing!")
    print("\nMissing files:")
    for f in missing_files:
        print(f"  - {f}")
    print("\n💡 Tip: Update the file paths in the configuration cell above")
else:
    print("\n✓ All files found! Ready to merge.")

## 4. Merge Function

Function to merge old and new data for a single station.

In [ ]:
def merge_era5_station_data(station_code, old_file, new_file, output_folder='merged_era5_data'):
    """
    Merge old and new ERA5 data for a single station
    """
    station_name = STATION_NAMES[station_code]
    print(f"\n{'='*80}")
    print(f"Processing: {station_name} ({station_code})")
    print('='*80)
    
    # Load old data
    print(f"Loading old data: {old_file}")
    try:
        df_old = pd.read_excel(old_file)
        df_old['datetime'] = pd.to_datetime(df_old['datetime'])
        print(f"  ✓ Loaded {len(df_old)} records ({df_old['datetime'].min().date()} to {df_old['datetime'].max().date()})")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return None
    
    # Load new data
    print(f"Loading new data: {new_file}")
    try:
        df_new = pd.read_excel(new_file)
        df_new['datetime'] = pd.to_datetime(df_new['datetime'])
        print(f"  ✓ Loaded {len(df_new)} records ({df_new['datetime'].min().date()} to {df_new['datetime'].max().date()})")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return None
    
    # Ensure same columns
    required_cols = ['datetime', 'heatindexmax2m', 'heatindexmin2m']
    df_old = df_old[required_cols]
    df_new = df_new[required_cols]
    
    # Check for overlap
    old_max_date = df_old['datetime'].max()
    new_min_date = df_new['datetime'].min()
    
    if old_max_date >= new_min_date:
        overlap_count = len(df_new[df_new['datetime'] <= old_max_date])
        print(f"  ⚠️ Overlap detected: removing {overlap_count} days from new data")
        df_new = df_new[df_new['datetime'] > old_max_date]
    
    # Merge
    print("Merging datasets...")
    df_merged = pd.concat([df_old, df_new], ignore_index=True)
    df_merged = df_merged.sort_values('datetime').reset_index(drop=True)
    
    # Remove duplicates
    duplicates = df_merged.duplicated(subset=['datetime']).sum()
    if duplicates > 0:
        print(f"  ⚠️ Removing {duplicates} duplicate dates")
        df_merged = df_merged.drop_duplicates(subset=['datetime'], keep='first')
    
    # Statistics
    print(f"\n✓ Merge complete:")
    print(f"  Total records: {len(df_merged)}")
    print(f"  Date range: {df_merged['datetime'].min().date()} to {df_merged['datetime'].max().date()}")
    print(f"  Missing data: Max={df_merged['heatindexmax2m'].isna().sum()}, Min={df_merged['heatindexmin2m'].isna().sum()}")
    print(f"  Max heat index: {df_merged['heatindexmax2m'].max():.1f}°F")
    print(f"  Min heat index: {df_merged['heatindexmin2m'].min():.1f}°F")
    
    # Save
    os.makedirs(output_folder, exist_ok=True)
    output_file = os.path.join(output_folder, f"{station_code}heatindex19712024_era5.xlsx")
    df_merged.to_excel(output_file, index=False)
    print(f"\n✓ Saved: {output_file}")
    
    return df_merged

print("✓ Merge function defined")

## 5. Quality Check Function

In [ ]:
def quality_check_era5_data(df_merged, station_code):
    """
    Perform quality checks on merged data
    """
    results = {
        'station': station_code,
        'total_records': len(df_merged),
        'date_range': f"{df_merged['datetime'].min().date()} to {df_merged['datetime'].max().date()}",
        'missing_max': df_merged['heatindexmax2m'].isna().sum(),
        'missing_min': df_merged['heatindexmin2m'].isna().sum(),
        'max_heat_index': df_merged['heatindexmax2m'].max(),
        'min_heat_index': df_merged['heatindexmin2m'].min(),
    }
    
    anomalies = []
    
    # Check for extreme values
    if df_merged['heatindexmax2m'].max() > 125:
        anomalies.append(f"Very high max: {df_merged['heatindexmax2m'].max():.1f}°F")
    
    if df_merged['heatindexmin2m'].min() < -40:
        anomalies.append(f"Very low min: {df_merged['heatindexmin2m'].min():.1f}°F")
    
    # Check for inversions
    inversions = (df_merged['heatindexmin2m'] > df_merged['heatindexmax2m']).sum()
    if inversions > 0:
        anomalies.append(f"{inversions} days where min > max")
    
    # Check for large jumps
    max_diff = df_merged['heatindexmax2m'].diff().abs()
    large_jumps = (max_diff > 30).sum()
    if large_jumps > 10:
        anomalies.append(f"{large_jumps} days with >30°F jump")
    
    results['anomalies'] = anomalies
    
    return results

print("✓ Quality check function defined")

## 6. Process All Stations

Merge data for all stations.

In [ ]:
print("="*80)
print("MERGING ALL STATIONS")
print("="*80)

merged_stations = []
failed_stations = []
quality_results = []
merged_dataframes = {}

for station_code in STATION_NAMES.keys():
    old_file = OLD_DATA_FILES[station_code]
    new_file = NEW_DATA_FILES[station_code]
    
    if os.path.exists(old_file) and os.path.exists(new_file):
        df_merged = merge_era5_station_data(station_code, old_file, new_file, OUTPUT_FOLDER)
        
        if df_merged is not None:
            merged_stations.append(station_code)
            merged_dataframes[station_code] = df_merged
            
            # Quality check
            qc_results = quality_check_era5_data(df_merged, station_code)
            quality_results.append(qc_results)
        else:
            failed_stations.append(station_code)
    else:
        print(f"\n✗ Skipping {station_code}: Missing files")
        failed_stations.append(station_code)

print(f"\n{'='*80}")
print("MERGE SUMMARY")
print('='*80)
print(f"\n✓ Successfully merged: {len(merged_stations)}/{len(STATION_NAMES)} stations")

if merged_stations:
    print("\nMerged stations:")
    for station in merged_stations:
        print(f"  ✓ {STATION_NAMES[station]} ({station})")

if failed_stations:
    print(f"\n✗ Failed: {len(failed_stations)} stations")
    for station in failed_stations:
        print(f"  ✗ {STATION_NAMES[station]} ({station})")

## 7. Quality Control Summary

Review quality checks for all stations.

In [ ]:
if quality_results:
    print("="*80)
    print("QUALITY CHECK SUMMARY")
    print("="*80)
    
    for qc in quality_results:
        print(f"\n{STATION_NAMES[qc['station']]} ({qc['station']}):")
        print(f"  Records: {qc['total_records']}")
        print(f"  Date range: {qc['date_range']}")
        print(f"  Missing data: Max={qc['missing_max']}, Min={qc['missing_min']}")
        print(f"  Extremes: Max={qc['max_heat_index']:.1f}°F, Min={qc['min_heat_index']:.1f}°F")
        
        if qc['anomalies']:
            print(f"  ⚠️ Anomalies:")
            for anomaly in qc['anomalies']:
                print(f"    - {anomaly}")
        else:
            print(f"  ✓ No anomalies detected")
else:
    print("No quality checks performed (no stations merged successfully)")

## 8. Visualize Merged Data (Optional)

Plot the complete time series for one station.

In [ ]:
import matplotlib.pyplot as plt

if merged_dataframes:
    # Plot one station as example
    station = 'KRDU'  # Change to any merged station
    
    if station in merged_dataframes:
        df = merged_dataframes[station]
        
        plt.figure(figsize=(16, 6))
        plt.plot(df['datetime'], df['heatindexmax2m'], label='Daily Max', alpha=0.5, linewidth=0.5)
        plt.plot(df['datetime'], df['heatindexmin2m'], label='Daily Min', alpha=0.5, linewidth=0.5)
        
        # Add vertical line at merge point (end of 2021)
        plt.axvline(pd.Timestamp('2021-12-31'), color='red', linestyle='--', alpha=0.5, label='Merge Point')
        
        plt.xlabel('Date')
        plt.ylabel('Heat Index (°F)')
        plt.title(f'{STATION_NAMES[station]} Heat Index (1971-2024)\nRed line shows where 1971-2021 and 2022-2024 data meet')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"\nSummary statistics for {STATION_NAMES[station]}:")
        print(df[['heatindexmax2m', 'heatindexmin2m']].describe())
    else:
        print(f"{station} not in merged data")
else:
    print("No data to visualize")

## Summary

### Files Created:
All files are in the `merged_era5_data/` folder:
- `KAVLheatindex19712024_era5.xlsx`
- `KRDUheatindex19712024_era5.xlsx`
- `KCLTheatindex19712024_era5.xlsx`
- `KGSOheatindex19712024_era5.xlsx`
- `KILMheatindex19712024_era5.xlsx`
- `KHSEheatindex19712024_era5.xlsx`

### Next Steps:
1. Review the quality check results above
2. Copy merged files to your dashboard project:
   ```bash
   cp merged_era5_data/*.xlsx /path/to/your/project/
   ```
3. Update and run your Streamlit dashboard
4. Dashboard will automatically:
   - Detect 1971-2024 date range
   - Recalculate percentiles with 54-year baseline
   - Update all visualizations

### Notes on ERA5:
- ERA5 is gridded reanalysis data (~25km resolution)
- Values may differ from station observations by 1-5°F
- This is normal and scientifically valid
- Trends should be consistent
- No missing data (complete coverage)